# 🎓 SmolSocrates-360M v2: Fine-Tuning a Socratic Coding Tutor

**Improved version** with:
- 🔥 Augmented dataset (~300+ examples vs 197)
- 🧠 Higher LoRA rank (r=32 vs r=16) for more capacity
- ⚡ Optimized training hyperparameters

⏱️ **Runtime**: ~20 minutes on a free T4 GPU

---

## 1. Setup

In [ ]:
# Install Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Load Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print(f"✅ Loaded {MODEL_NAME}")

## 3. Apply LoRA (r=32, doubled capacity)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # ← doubled from 16
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=64,  # ← doubled from 32 (maintains ratio)
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"🔧 LoRA r=32 applied! Trainable: {trainable:,} ({100 * trainable / total:.2f}%)"
)

## 4. Load Augmented Dataset

Upload `train_data_augmented.jsonl` and `eval_data.jsonl` to Colab first:
- Click the 📁 folder icon on the left
- Upload both files from your `data/` folder

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a Socratic coding tutor. Your role is to help students learn programming "
    "by asking guiding questions — never by giving direct answers or complete solutions.\n\n"
    "Rules:\n"
    "1. NEVER provide complete code solutions or direct fixes.\n"
    "2. Ask targeted questions that lead the student to discover the answer themselves.\n"
    "3. Break complex problems into smaller, manageable steps.\n"
    "4. If a student is stuck, offer a progressive hint — start broad, get specific.\n"
    "5. Celebrate progress and encourage self-discovery.\n"
    "6. Match the student's technical level in your language.\n"
    "7. When debugging, ask the student to trace through their code mentally."
)


# Load from uploaded files
def load_jsonl(path):
    examples = []
    with open(path) as f:
        for line in f:
            if line.strip():
                examples.append(json.loads(line))
    return examples


# Try augmented dataset first, fall back to HuggingFace
try:
    train_raw = load_jsonl("train_data_augmented.jsonl")
    print(f"📥 Loaded augmented dataset: {len(train_raw)} examples")
except FileNotFoundError:
    print("⚠️ train_data_augmented.jsonl not found, falling back to HuggingFace dataset")
    from datasets import load_dataset

    ds = load_dataset("AndreiSobo/PACT-Socratic-Coding-Tutor", split="train")
    train_raw = [{"messages": ex["messages"]} for ex in ds]
    print(f"📥 Loaded from HuggingFace: {len(train_raw)} examples")

try:
    eval_raw = load_jsonl("eval_data.jsonl")
    print(f"📥 Loaded eval dataset: {len(eval_raw)} examples")
except FileNotFoundError:
    print("⚠️ eval_data.jsonl not found, creating split from training data")
    import random

    random.seed(42)
    random.shuffle(train_raw)
    eval_raw = train_raw[:30]
    train_raw = train_raw[30:]
    print(f"   Split: {len(train_raw)} train / {len(eval_raw)} eval")

# Convert to HF Dataset
train_dataset = Dataset.from_list([{"messages": ex["messages"]} for ex in train_raw])
eval_dataset = Dataset.from_list([{"messages": ex["messages"]} for ex in eval_raw])
print(f"\n✅ Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

## 5. Baseline Evaluation

In [ ]:
import re


def quick_socratic_score(response: str) -> dict:
    """Score Socratic quality (0-9)."""
    code_blocks = len(re.findall(r"```[\s\S]*?```", response))
    no_code_score = 3 if code_blocks == 0 else (1 if code_blocks == 1 else 0)

    questions = [
        s for s in re.split(r"[.!\n]", response) if "?" in s and len(s.strip()) > 10
    ]
    question_score = min(3, len(questions))

    encouraging = [
        "great",
        "good",
        "think about",
        "consider",
        "let's",
        "try",
        "what if",
    ]
    enc_count = sum(1 for p in encouraging if p in response.lower())
    enc_score = min(3, enc_count)

    return {
        "no_code": no_code_score,
        "questions": question_score,
        "encouragement": enc_score,
        "total": no_code_score + question_score + enc_score,
        "max": 9,
        "code_blocks": code_blocks,
        "num_questions": len(questions),
    }


TEST_PROMPTS = [
    "How do I reverse a string in Python?",
    "My for loop never stops running. Here's my code:\n```python\ni = 0\nwhile i < 10:\n    print(i)\n```\nWhat's wrong?",
    "What's the difference between a list and a tuple in Python?",
    "I'm getting an IndexError in my code. How do I fix it?",
    "Can you explain recursion to me? I don't get it.",
]


def run_eval(model, tokenizer, label="Model"):
    """Run evaluation and return results."""
    FastLanguageModel.for_inference(model)
    results = []
    for i, prompt in enumerate(TEST_PROMPTS):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=256,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
            )
        response = tokenizer.decode(
            outputs[0][inputs.shape[1] :], skip_special_tokens=True
        )
        score = quick_socratic_score(response)
        results.append({"prompt": prompt, "response": response, "score": score})
        print(f"  [{i + 1}] Score: {score['total']}/9 | Q: {prompt[:60]}...")
    avg = sum(r["score"]["total"] for r in results) / len(results)
    print(f"  📊 {label} Average: {avg:.1f}/9")
    return results, avg


print("=" * 60)
print("📊 BASELINE EVALUATION")
print("=" * 60)
baseline_results, baseline_avg = run_eval(model, tokenizer, "Baseline")

## 6. Fine-Tune (Optimized v2)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

FastLanguageModel.for_training(model)


# Pre-process dataset: convert messages → text
def convert_to_text(example):
    msgs = example["messages"]
    if isinstance(msgs, dict):
        converted = [
            {"role": r, "content": c} for r, c in zip(msgs["role"], msgs["content"])
        ]
    else:
        converted = msgs
    text = tokenizer.apply_chat_template(
        converted, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


train_text = train_dataset.map(convert_to_text)
eval_text = eval_dataset.map(convert_to_text)

print(f"✅ Processed {len(train_text)} train, {len(eval_text)} eval examples")
print(f"\nSample (first 200 chars):")
print(train_text[0]["text"][:200])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=10,
        warmup_steps=10,
        learning_rate=5e-4,
        lr_scheduler_type="cosine",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        output_dir="outputs",
        save_strategy="no",
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=42,
        report_to="none",
    ),
)

print("\n🚀 Starting training...")
stats = trainer.train()
print(f"\n✅ Training complete!")
print(
    f"   Steps: {stats.global_step} | Loss: {stats.training_loss:.4f} | Time: {stats.metrics['train_runtime']:.0f}s"
)

## 7. Post-Training Evaluation

In [ ]:
print("=" * 60)
print("📊 FINE-TUNED EVALUATION")
print("=" * 60)
finetuned_results, finetuned_avg = run_eval(model, tokenizer, "Fine-tuned")

## 8. Before vs After

In [ ]:
print("\n" + "=" * 70)
print("📊 BEFORE vs AFTER COMPARISON")
print("=" * 70)
print(f"\n{'Metric':<25s} {'Baseline':>10s} {'Fine-tuned':>12s} {'Delta':>8s}")
print(f"{'-' * 25} {'-' * 10} {'-' * 12} {'-' * 8}")

delta = finetuned_avg - baseline_avg
sign = "+" if delta > 0 else ""
print(
    f"{'Overall (avg)':<25s} {baseline_avg:>10.1f} {finetuned_avg:>12.1f} {sign}{delta:>7.1f}"
)

for dim in ["no_code", "questions", "encouragement"]:
    b = sum(r["score"][dim] for r in baseline_results) / len(baseline_results)
    f = sum(r["score"][dim] for r in finetuned_results) / len(finetuned_results)
    d = f - b
    s = "+" if d > 0 else ""
    print(f"{dim:<25s} {b:>10.1f} {f:>12.1f} {s}{d:>7.1f}")

b_ncr = (
    sum(1 for r in baseline_results if r["score"]["code_blocks"] == 0)
    / len(baseline_results)
    * 100
)
f_ncr = (
    sum(1 for r in finetuned_results if r["score"]["code_blocks"] == 0)
    / len(finetuned_results)
    * 100
)
d = f_ncr - b_ncr
s = "+" if d > 0 else ""
print(f"{'No-Code Rate (%)':<25s} {b_ncr:>10.0f} {f_ncr:>12.0f} {s}{d:>7.0f}")

print("\n" + "=" * 70)
print("\n📝 SIDE-BY-SIDE:")
for i in range(min(3, len(TEST_PROMPTS))):
    print(f"\n{'─' * 60}")
    print(f"Q: {TEST_PROMPTS[i][:80]}")
    print(f"\n🔴 BASELINE ({baseline_results[i]['score']['total']}/9):")
    print(baseline_results[i]["response"][:250])
    print(f"\n🟢 FINE-TUNED ({finetuned_results[i]['score']['total']}/9):")
    print(finetuned_results[i]["response"][:250])

## 9. Push to HuggingFace

In [ ]:
from huggingface_hub import login

login()  # Interactive login

In [ ]:
# ✏️ CHANGE THIS
HF_USERNAME = "your-username"
MODEL_REPO = f"{HF_USERNAME}/SmolSocrates-360M"

print(f"📤 Pushing to {MODEL_REPO}...")
model.push_to_hub(MODEL_REPO)
tokenizer.push_to_hub(MODEL_REPO)
print(f"✅ https://huggingface.co/{MODEL_REPO}")

## 10. Save Results

In [ ]:
import json

results = {
    "model_name": MODEL_REPO,
    "base_model": MODEL_NAME,
    "lora_config": {"r": 32, "lora_alpha": 64},
    "training": {
        "dataset": "PACT + synthetic augmentation",
        "train_examples": len(train_dataset),
        "eval_examples": len(eval_dataset),
        "epochs": 10,
        "learning_rate": 5e-4,
    },
    "baseline_avg": baseline_avg,
    "finetuned_avg": finetuned_avg,
    "improvement": finetuned_avg - baseline_avg,
    "baseline_results": baseline_results,
    "finetuned_results": finetuned_results,
}

with open("evaluation_results_v2.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"💾 Saved to evaluation_results_v2.json")
print(
    f"🎯 Improvement: {baseline_avg:.1f} → {finetuned_avg:.1f} ({finetuned_avg - baseline_avg:+.1f})"
)

In [ ]:
from google.colab import files

files.download("evaluation_results_v2.json")